# ODOT SCD Components — Functional Verification

This notebook exercises every ODOT Standard Construction Drawing (SCD)
component civilpy has built (`src/civilpy/structural/odot/*.py`): for each
one, it calls the catalog lookup(s) and the pure-Python `layout_*()`
generator with representative inputs, and checks the result against the
transcribed drawing data (spacing, counts, formulas, guarded lookups).

This is **not** a replacement for `pytest tests/structural` (which is the
authoritative, CI-run test suite — see `docs/SCD_BUILD_LOG.md` for the
per-SCD test counts) — it's a single narrative pass over the whole SCD
program so a person (or a future session) can see every component's
inputs/outputs side by side, the way the "Rhino → MIDAS Pipeline
Verification" notebook in this same folder does for the girder pipeline.

Everything here runs **offline** — no Rhino needed, since all engineering
content lives in pure-Python `civilpy.structural.odot.*` modules. The
Grasshopper (`Notebooks/res/*.py`) scripts are thin drawing layers over
these same functions and are not re-tested here (they need a live Rhino
8 session with `rhino3dm`/`Rhino.Geometry` available).

**2026-07-12 extension** — added the Wave-8/9 railing components (roadway barriers + curbs, RM-4.6 end sections, RM-4.4 transitions, RM-5.2 bikeway railing, RM-4.7 PCB thrie-beam transitions, MGS terminals), the PSID-1-13 prestressed I-beam slice (catalog, strand designer, MIDAS spoke, BrIM emit), and the **integration sections**: steel-girder / box-beam / PS I-beam BrIM emits with pay-item rollups, the Phase 4/4v substructure type gallery, and every barrier shape family generated to a tagged `.3dm` (this last section needs `rhino3dm`, which the layer-taxonomy section already required).


In [1]:
def check(name, ok, detail=""):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, name

def approx(a, b, tol=1e-6):
    """True if a and b agree within a relative/absolute tolerance -- a
    dependency-free stand-in for pytest.approx for this narrative notebook."""
    return abs(a - b) <= tol * max(1.0, abs(b))

report = {}

""" General TODOs
# //TODO - Most of the imported civilpy libraries based on these SCDs or AASHTO 
specs need to take in "des_yr" as an optional input, many of the AASHTO Checks 
already do. They're all being designed with the most modern SCD we currently 
have `des_yr=2026` but a long term goal will be to be able to support quickly 
generating structures that have been built previously/are exisiting.
""";

## A-1-20 — Typical Abutment Detail (Expansion Joints)

`odot.typical_abutment`: **guidance only** (the sheet's own note says not
to use it as a standalone construction drawing) — the bearing-seat and
wingwall-limit formulas, section minimums, for a visual check only.

In [2]:
from civilpy.structural.odot.typical_abutment import (
    AbutmentInput as TypicalAbutmentInput, bearing_seat_dim_a_ft, layout_typical_abutment,
)

"""
# //TODO - Should mostly be handled by the TODOs in Notebooks/Rhino Components/A-1-20.py but
Missing a lot of inputs on these functions, Near/Far Abutment determines which 
way the wingwalls would be swept, probably needs to take in an alignment and a 
terrain model to be able to understand what kind of slope is required on the 
wingwall itself. That's a large task to build. Terrain models can be automatically
generated from OGRIP Data, I think CivilPy Already has existing infrastructure to
handle that. The rebar isn't coming through in the Rhino model but appears to exist
in the python object when it's inspected. It should probably be independent of any 
approach slab values, I'm not sure why it's depicted as having it inherited by composition.
Not able to tell if any (and which if any are) design checks are being implemented when
this function is being utilized, probably one of the most difficult standards to successfully
implement.
"""

check("A-1-20: DIM.A grows with skew", bearing_seat_dim_a_ft(30.0) > bearing_seat_dim_a_ft(0.0))

layout = layout_typical_abutment(TypicalAbutmentInput(
    width_ft=30.0, skew_deg=15.0, wingwall_length_ft=6.0, footing_depth_ft=3.0,
    backwall_height_ft=5.0,
))
check("A-1-20: wingwall springs from backwall end", layout.wingwall_outline[0] == layout.backwall_outline[2])
report["A-1-20"] = 1

  [PASS] A-1-20: DIM.A grows with skew
  [PASS] A-1-20: wingwall springs from backwall end


## AS-1-15 — Reinforced Concrete Approach Slab

`odot.approach_slab`: the reinforcing-steel table (spans 15/20/25/30 ft),
bracketed bar-count formulas, seat/joint catalog, and `layout_approach_slab`.

In [3]:
from civilpy.structural.odot.approach_slab import (
    ApproachSlabInput, approach_slab_design, layout_approach_slab,
)

design = approach_slab_design(25.0)
check("AS-1-15: 25 ft design found", design.length_ft == 25.0)

layout = layout_approach_slab(ApproachSlabInput(
    length_ft=25.0, width_ft=40.0, skew_deg=15.0, end_thickness_in=design.thickness_in,
))
check("AS-1-15: profile has no duplicate points", len(layout.profile) == len(set(layout.profile)))
check("AS-1-15: bars generated", len(layout.bars) > 0)
marks = {b.mark for b in layout.bars}
check("AS-1-15: bottom (A) and B501 bar marks present", {"A1003", "B501"}.issubset(marks))
report["AS-1-15"] = len(layout.bars)


  [PASS] AS-1-15: 25 ft design found
  [PASS] AS-1-15: profile has no duplicate points
  [PASS] AS-1-15: bars generated
  [PASS] AS-1-15: bottom (A) and B501 bar marks present


## DS-1-92 — Stainless Steel Drip Strip

`odot.drip_strip`: railing-dependent placement catalog, perforation pattern,
and `drip_strip_runs` (fascia run generator).

In [4]:
from civilpy.structural.odot.drip_strip import (
    PLACEMENTS, drip_strip_runs, placement, strip_profile_in,
)

p = placement("DBR-2-73")
check("DS-1-92: DBR-2-73 placement found", p.railing == "DBR-2-73")

runs = drip_strip_runs(120.0, (10.0, 30.0, 50.0, 70.0, 90.0, 110.0), "DBR-2-73")
check("DS-1-92: runs generated", len(runs) > 0)
prof = strip_profile_in("upper", bent=True)
check("DS-1-92: bent profile has points", len(prof) >= 3)
report["DS-1-92"] = len(runs)


  [PASS] DS-1-92: DBR-2-73 placement found
  [PASS] DS-1-92: runs generated
  [PASS] DS-1-92: bent profile has points


## PCB-91 — Portable Concrete Barrier

`odot.portable_barrier`: the NJ-shape section profile, segment/joint/anchor
layout (crash levels stay in `bridge_railing`).

In [5]:
from civilpy.structural.odot.portable_barrier import (
    anchor_hole_stations_ft, barrier_run, profile_points_in, run_length_ft,
)

prof = profile_points_in(chamfered=True)
check("PCB-91: profile is closed/mirror-symmetric", prof[0][0] == -prof[-1][0])

run = barrier_run(4, segment_length_ft=10.0, joint_gap_in=0.25)
check("PCB-91: 4-segment run built", len(run) == 4)
check("PCB-91: run length matches segments+gaps", approx(run_length_ft(run), 4*10.0 + 3*0.25/12.0))
anchors = anchor_hole_stations_ft(10.0)
check("PCB-91: anchor hole stations found", len(anchors) > 0)
report["PCB-91"] = len(run)


  [PASS] PCB-91: profile is closed/mirror-symmetric
  [PASS] PCB-91: 4-segment run built
  [PASS] PCB-91: run length matches segments+gaps
  [PASS] PCB-91: anchor hole stations found


## AS-2-15 — Approach Slab Installation (Sleeper Slab)

`odot.sleeper_slab`: Type A/C reinforced concrete sleeper slab under the
approach-slab/pavement joint (Type B has none — raises `ValueError`).

In [6]:
from civilpy.structural.odot.sleeper_slab import SleeperSlabInput, layout_sleeper_slab

layout = layout_sleeper_slab(SleeperSlabInput(width_ft=24.0, skew_deg=20.0, installation="A"))
check("AS-2-15: outline is a parallelogram (4 pts)", len(layout.outline) == 4)
check("AS-2-15: SS501/SS502 bars present", {"SS501", "SS502"}.issubset({b.mark for b in layout.bars}))

try:
    layout_sleeper_slab(SleeperSlabInput(width_ft=24.0, installation="B"))
    check("AS-2-15: Type B correctly has no sleeper slab", False)
except ValueError as exc:
    check("AS-2-15: Type B raises (no sleeper slab)", "Type B" in str(exc))
report["AS-2-15"] = len(layout.bars)


  [PASS] AS-2-15: outline is a parallelogram (4 pts)
  [PASS] AS-2-15: SS501/SS502 bars present
  [PASS] AS-2-15: Type B raises (no sleeper slab)


## HW-2.1 / HW-2.2 — Half-Height Headwalls

`odot.headwall`: circular-pipe headwall (end treatment "A") for corrugated
metal/plastic pipe (HW-2.1) and concrete pipe (HW-2.2, `concrete=True`).

In [7]:
from civilpy.structural.odot.headwall import HeadwallInput, layout_headwall

hw21 = layout_headwall(HeadwallInput(diameter_in=36.0, concrete=False))
check("HW-2.1: pipe opening below cover minimum satisfied", hw21.cover_in >= 6.0)

hw22 = layout_headwall(HeadwallInput(diameter_in=36.0, concrete=True))
check("HW-2.2: concrete-pipe table used", hw22.table is not hw21.table or hw22.concrete_cy != hw21.concrete_cy)

try:
    layout_headwall(HeadwallInput(diameter_in=60.0, concrete=False))
    check("HW-2.1: D=60 in should exceed treatment-A cover limit", False)
except ValueError as exc:
    check("HW-2.1: D=60 in correctly raises (end treatment B not modeled)", "treatment" in str(exc).lower())
report["HW-2.1/HW-2.2"] = 2


  [PASS] HW-2.1: pipe opening below cover minimum satisfied
  [PASS] HW-2.2: concrete-pipe table used
  [PASS] HW-2.1: D=60 in correctly raises (end treatment B not modeled)


## HW-1.1 — Full-Height Headwalls

`odot.full_height_headwall`: pipe-diameter x skew-angle dimension/quantity
table (42-84 in, skew 0-45 deg); Type A symmetric / Type B asymmetric
wingwalls, skew bucket pinned to the sheet's own 10 deg Type A/B cutoff.

In [8]:
from civilpy.structural.odot.full_height_headwall import (
    HeadwallInput as FHHInput, layout_full_height_headwall, nearest_skew_bucket,
)

check("HW-1.1: 9 deg skew -> Type A bucket (0)", nearest_skew_bucket(9.0) == 0.0)
check("HW-1.1: 11 deg skew -> Type B bucket (15)", nearest_skew_bucket(11.0) == 15.0)

square = layout_full_height_headwall(FHHInput(diameter_in=60.0, skew_deg=0.0))
check("HW-1.1: Type A is symmetric", approx(square.wing1[2][0], -square.wing2[2][0]))

skewed = layout_full_height_headwall(FHHInput(diameter_in=60.0, skew_deg=30.0))
check("HW-1.1: Type B wingwalls are asymmetric", skewed.type_ == "B")
report["HW-1.1"] = 2


  [PASS] HW-1.1: 9 deg skew -> Type A bucket (0)
  [PASS] HW-1.1: 11 deg skew -> Type B bucket (15)
  [PASS] HW-1.1: Type A is symmetric
  [PASS] HW-1.1: Type B wingwalls are asymmetric


## BCHW — Precast Box Culvert Headwall/Wingwall

`odot.box_culvert_headwall`: a detailing template (every overall dimension
is project-supplied, no catalog) plus the TYPE-1..TYPE-8 rebar bend legend.

In [9]:
from civilpy.structural.odot.box_culvert_headwall import (
    WingwallInput, bend_shape, layout_wingwall,
)

pts = bend_shape("TYPE-8", A=12.0, B=18.0, skew_deg=20.0)
check("BCHW: TYPE-8 corner bar bend generated", len(pts) == 3)

layout = layout_wingwall(WingwallInput(
    length_ft=10.0, skew_deg=15.0, wall_height_ft=8.0, foreslope_height_ft=4.0,
    cutoff_wall_height_ft=2.0, footing_width_ft=6.0, box_wall_thickness_in=12.0,
))
check("BCHW: wingwall footprint generated", len(layout.wingwall_outline) == 4)
report["BCHW"] = len(pts)


  [PASS] BCHW: TYPE-8 corner bar bend generated
  [PASS] BCHW: wingwall footprint generated


## Bridge Railings (SBR/BR/TST/DBR/TBR/PCB) — the shared `build_barriers()` pipeline

`odot.bridge_railing` already cataloged every Wave-3 railing; the generic
`rhino_barrier.shape_family()`/`barrier_profile()` dispatch draws any of
them. This section re-verifies the **BR-2-15 fix**: the "combination"
family (full-height concrete barrier + steel tube rail on top) used to be
misclassified as a bare "steel tube" railing because its shape string
contains the substring "tube".

In [10]:
from civilpy.structural.odot.bridge_railing import BRIDGE_RAILINGS, railing
from civilpy.structural.rhino_barrier import barrier_profile, shape_family

for name in ("BR-1 (36 in)", "SBR-1 (42 in)", "TST-2 (three steel tube)",
             "PCB (portable, unanchored)", "BR-2 (sidewalk barrier + twin tube)"):
    r = railing(name)
    fam = shape_family(r)
    print(f"  {name:38s} -> {fam}")

br2 = railing("BR-2 (sidewalk barrier + twin tube)")
check("BR-2-15: classified as 'combination', not 'steel tube'", shape_family(br2) == "combination")
prof = barrier_profile(br2, 42.0 / 12.0, side=+1)
check("BR-2-15: full 42 in barrier height (not a 10 in curb)",
      approx(max(z for _, z in prof), 3.5))
report["bridge_railing"] = len(BRIDGE_RAILINGS)


  BR-1 (36 in)                           -> new jersey
  SBR-1 (42 in)                          -> single slope
  TST-2 (three steel tube)               -> steel tube
  PCB (portable, unanchored)             -> portable
  BR-2 (sidewalk barrier + twin tube)    -> combination
  [PASS] BR-2-15: classified as 'combination', not 'steel tube'
  [PASS] BR-2-15: full 42 in barrier height (not a 10 in curb)


## SB-1-24 — Single Span Slab Bridges

`odot.slab_bridge`: full SLAB DATA + EDGE BEAM SLAB DATA tables (spans
11-38 ft), skewed parallelogram plan, A/B/M/N longitudinal bar mats.

In [11]:
from civilpy.structural.odot.slab_bridge import SlabBridgeInput, layout_slab_bridge

layout = layout_slab_bridge(SlabBridgeInput(span_ft=24, width_ft=30.0, skew_deg=15.0))
check("SB-1-24: thickness matches the table", layout.thickness_in == 18.25)
check("SB-1-24: all 4 bar marks present", {"A", "B", "M", "N"} == {b.mark for b in layout.bars})
check("SB-1-24: bridge length > span (skew adds length)", layout.bridge_length_ft > 24.0)
report["SB-1-24"] = len(layout.bars)


  [PASS] SB-1-24: thickness matches the table
  [PASS] SB-1-24: all 4 bar marks present
  [PASS] SB-1-24: bridge length > span (skew adds length)


## CS-1-24 — Continuous Slab Bridges

`odot.continuous_slab_bridge`: the largest table in the SCD program
(779 numeric entries, end spans 14-46 ft, interior span fixed at 1.25x
end span); A/B/C/D/E longitudinal bar mats, two piers.

In [12]:
from civilpy.structural.odot.continuous_slab_bridge import (
    ContinuousSlabInput, interior_span_ft, layout_continuous_slab,
)

check("CS-1-24: interior span is 1.25x end span", approx(interior_span_ft(24), 30.0))

layout = layout_continuous_slab(ContinuousSlabInput(end_span_ft=24, width_ft=30.0, skew_deg=10.0))
check("CS-1-24: total length = 2*end + interior",
      approx(layout.total_length_ft, 2*24 + interior_span_ft(24)))
check("CS-1-24: two pier stations", len(layout.pier_stations) == 2)
check("CS-1-24: E-bars present (span 24 >= 22)", "E" in {b.mark for b in layout.bars})
report["CS-1-24"] = len(layout.bars)


  [PASS] CS-1-24: interior span is 1.25x end span
  [PASS] CS-1-24: total length = 2*end + interior
  [PASS] CS-1-24: two pier stations
  [PASS] CS-1-24: E-bars present (span 24 >= 22)


## CPA-1-08 — Capped Pile Abutment (Slab Bridges)

`odot.capped_pile_abutment`: SB-1-24's companion. A detailing template
(overall dimensions project-supplied) plus the TYPE-1..TYPE-5 rebar bend
legend (TYPE-6/D801 cross-references `approach_slab`'s own D801 bar).

In [13]:
from civilpy.structural.odot.capped_pile_abutment import (
    AbutmentInput, layout_capped_pile_abutment, rebar_mark,
)

d801 = rebar_mark("D801")[0]
check("CPA-1-08: D801 cross-references approach_slab", "approach_slab" in d801.note)

layout = layout_capped_pile_abutment(AbutmentInput(
    wingwall_length_ft=10.0, skew_deg=15.0, n_piles=6, pile_spacing_ft=4.0,
    footing_depth_ft=3.0,
))
check("CPA-1-08: 6 pile points generated", len(layout.pile_points) == 6)
check("CPA-1-08: wingwall springs from cap end", layout.wingwall_outline[0] == layout.cap_outline[2])
report["CPA-1-08"] = len(layout.pile_points)


  [PASS] CPA-1-08: D801 cross-references approach_slab
  [PASS] CPA-1-08: 6 pile points generated
  [PASS] CPA-1-08: wingwall springs from cap end


## CPP-1-08 — Capped Pile Pier (Continuous Slab Bridges)

`odot.capped_pile_pier`: CS-1-24's companion. Genuinely parametric (unlike
CPA-1-08/BCHW) — the sheet's own pier-length formula, fixed cap width/
end-radius; only pile count/spacing stay project-supplied.

In [14]:
from civilpy.structural.odot.capped_pile_pier import (
    PierInput, layout_capped_pile_pier, pier_length_ft,
)

L = pier_length_ft(30.0, 15.0)
check("CPP-1-08: skewed pier is longer than square (secant term)", L > pier_length_ft(30.0, 0.0))

layout = layout_capped_pile_pier(PierInput(
    slab_width_ft=30.0, skew_deg=15.0, n_piles=6, pile_spacing_ft=5.0,
))
check("CPP-1-08: cap outline is a closed stadium shape", len(layout.cap_outline) > 4)
check("CPP-1-08: 6 pile points generated", len(layout.pile_points) == 6)
report["CPP-1-08"] = len(layout.pile_points)


  [PASS] CPP-1-08: skewed pier is longer than square (secant term)
  [PASS] CPP-1-08: cap outline is a closed stadium shape
  [PASS] CPP-1-08: 6 pile points generated


## RB-1-55 — Rockers and Bolsters

`odot.rocker_bolster`: F/R capacity table (75-300 kips) plus
`layout_rocker_bolster` — tapered body, flat top (bolster) or curved-top
(rocker, TOP BEARING DETAIL radius formula).

In [15]:
from civilpy.structural.odot.rocker_bolster import (
    layout_rocker_bolster, rocker_bolster, top_bearing_plate_radius_in,
)

rb = rocker_bolster(150)
layout = layout_rocker_bolster(rb)
check("RB-1-55: bolster top narrower than base",
      (layout.bolster_top[1][0]-layout.bolster_top[0][0]) < (layout.base_outline[1][0]-layout.base_outline[0][0]))
check("RB-1-55: rocker radius derived from A",
      approx(layout.rocker_top_radius_in, top_bearing_plate_radius_in(rb.dims["A"])))

r75 = rocker_bolster(75)
check("RB-1-55: R-75 has no matching bolster", r75.bolster_no == "")
report["RB-1-55"] = 1


  [PASS] RB-1-55: bolster top narrower than base
  [PASS] RB-1-55: rocker radius derived from A
  [PASS] RB-1-55: R-75 has no matching bolster


## FB-1-82 — Fixed Bearings for Steel Beam and Girder Bridges

`odot.fixed_bearing`: F-50..F-400 pin-bearing table; `layout_fixed_bearing`
builds a self-consistent stack (base plate -> pin -> top plate).

In [16]:
from civilpy.structural.odot.fixed_bearing import fixed_bearing, layout_fixed_bearing

fb = fixed_bearing("F-150")
layout = layout_fixed_bearing(fb)
pin_bottom = layout.pin_center[2] - layout.pin_diameter_in / 2.0
check("FB-1-82: pin sits on top of the base plate (no overlap)", pin_bottom >= layout.base_thickness_in - 1e-9)
check("FB-1-82: top plate above the pin", layout.top_z_in > layout.pin_center[2])

f400 = fixed_bearing("F-400")
check("FB-1-82: F-400 requires bearing stiffeners", f400.stiffeners_required)
report["FB-1-82"] = 1


  [PASS] FB-1-82: pin sits on top of the base plate (no overlap)
  [PASS] FB-1-82: top plate above the pin
  [PASS] FB-1-82: F-400 requires bearing stiffeners


## BD-1-11 — Bearing Details for Box Beam Bridges

`odot.box_beam`'s `BeveledLoadPlate`/`load_plate_bevel` (pre-existing) plus
the new `layout_load_plate`, sized to a B1/B2 bearing pad and tilted to
the roadway grade/skew.

In [17]:
from civilpy.structural.odot.box_beam import bearing_pad, layout_load_plate

pad = bearing_pad("B1")
layout = layout_load_plate("B1", longitudinal_grade=0.04, skew_deg=20.0)
length = layout.bottom_face[1][0] - layout.bottom_face[0][0]
check("BD-1-11: plate sized to the B1 pad footprint", approx(length, pad.length))
zs = [p[2] for p in layout.top_face]
check("BD-1-11: plate top tilts with grade+skew", len(set(round(z, 6) for z in zs)) > 1)
report["BD-1-11"] = 1


  [PASS] BD-1-11: plate sized to the B1 pad footprint
  [PASS] BD-1-11: plate top tilts with grade+skew


## EXJ-4-87 / EXJ-5-93 — Strip Seal Expansion Joints

Detailing templates for a manufacturer-generic strip-seal gland:
EXJ-4-87 (steel stringers) tabulates the a1-a4 support-angle formulas;
EXJ-5-93 (box beams) tabulates the plate "A"/"B"/"C" spacing + joint-
length formula.

In [18]:
from civilpy.structural.odot.strip_seal_joint import (
    StripSealJointInput, layout_strip_seal_joint,
)
from civilpy.structural.odot.strip_seal_joint_box_beam import (
    BoxBeamJointInput, layout_box_beam_joint,
)

steel = layout_strip_seal_joint(StripSealJointInput(
    width_ft=30.0, skew_deg=20.0, stringer_stations_ft=(0.0, 7.0, 14.0, 21.0, 28.0),
    top_flange_width_in=12.0,
))
check("EXJ-4-87: one support-angle run per stringer", len(steel.support_angles) == 5)

box = layout_box_beam_joint(BoxBeamJointInput(n_beams=5, beam_width_in=48.0, skew_deg=20.0))
check("EXJ-5-93: 4 beam-gap stations for 5 beams", len(box.beam_gap_stations_ft) == 4)
report["EXJ-4-87/EXJ-5-93"] = len(steel.support_angles) + len(box.beam_gap_stations_ft)


  [PASS] EXJ-4-87: one support-angle run per stringer
  [PASS] EXJ-5-93: 4 beam-gap stations for 5 beams


## Rhino layer taxonomy — `civilpy.structural.rhino_layers`

Confirms the shared layer-path constants match `Core/Gdr.cs`
(RhinoODOTExtension, commit `34f7051`) exactly, and that `ensure_layer`
builds the same nested tree in an offline `rhino3dm.File3dm` that the C#
plugin builds in a live document.

In [19]:
import rhino3dm
from civilpy.structural import rhino_layers as rl

f = rhino3dm.File3dm()
idx = rl.ensure_layer(f, rl.LAYER_BOX_BEAMS)
check("rhino_layers: Superstructure::Box Beams created", f.Layers[idx].FullPath == "Superstructure::Box Beams")
idx2 = rl.ensure_layer(f, rl.LAYER_GIRDERS)
parents = {l.FullPath for l in f.Layers if l.FullPath == "Superstructure"}
check("rhino_layers: Superstructure parent shared across leaves", len(parents) == 1)
report["rhino_layers"] = len(list(f.Layers))


  [PASS] rhino_layers: Superstructure::Box Beams created


  [PASS] rhino_layers: Superstructure parent shared across leaves


## RM-4.3 / RM-4.5 / RM-4.8 / RM-4.9 — Roadway Single Slope Barriers, and BP-5.1 Curbs, RM-4.1/RM-4.2 Portable Barriers

The Office of Roadway Engineering counterparts of the bridge parapets:
freestanding single-slope barriers (Types B/B1/C/C1/D/N, plus the
moment-slab Type E whose face the sheet leaves undimensioned), the
BP-5.1 concrete curb family, and the two roadway portable concrete
barriers.

In [20]:
from civilpy.structural.odot.roadway_barrier import (
    ROADWAY_BARRIERS, RoadwayBarrierInput, layout_roadway_barrier)
from civilpy.structural.odot.concrete_curb import CURB_TYPES, curb_height_in, curb_profile_in
from civilpy.structural.odot.roadway_portable_barrier import (
    ROADWAY_PORTABLE_BARRIERS, TRANSITION_50_TO_32)

for des in ("Type B", "Type B1", "Type D", "Type N"):
    lay = layout_roadway_barrier(RoadwayBarrierInput(des, 100.0))
    h = max(z for _, z in lay.profile)
    w = max(o for o, _ in lay.profile) - min(o for o, _ in lay.profile)
    print(f"  {des:8s}: {h:5.2f} in tall x {w:6.3f} in base, symmetric single slope")
check("RM-4.3 Type B profile is 42 x 28 in",
      max(z for _, z in layout_roadway_barrier(RoadwayBarrierInput('Type B', 10)).profile) == 42.0)

# Type E's concrete face is not dimensioned on RM-4.9 -- the layout refuses
try:
    layout_roadway_barrier(RoadwayBarrierInput("Type E", 10.0))
    check("RM-4.9 Type E refuses to invent an envelope", False)
except ValueError:
    check("RM-4.9 Type E refuses to invent an envelope", True)

prof = curb_profile_in("Type 2-A")
check("BP-5.1 Type 2-A curb profile closes", len(prof) >= 4,
      f"h = {curb_height_in('Type 2-A'):g} in")
check("BP-5.1 catalog resolves every sheet label", len(CURB_TYPES) == 19)

pcb32 = ROADWAY_PORTABLE_BARRIERS["RM Portable (32 in, pin & loop)"]
check("RM-4.2 32 in PCB cataloged", pcb32.height == 32.0)
check("RM-4.1 50->32 transition is 6 ft",
      TRANSITION_50_TO_32.length_ft == 6.0)
report["roadway_barrier"] = len(ROADWAY_BARRIERS)
report["concrete_curb"] = len(CURB_TYPES)
report["roadway_pcb"] = len(ROADWAY_PORTABLE_BARRIERS)


  Type B  : 42.00 in tall x 28.000 in base, symmetric single slope
  Type B1 : 57.00 in tall x 33.750 in base, symmetric single slope
  Type D  : 42.00 in tall x 28.000 in base, symmetric single slope
  Type N  : 81.00 in tall x 42.875 in base, symmetric single slope
  [PASS] RM-4.3 Type B profile is 42 x 28 in
  [PASS] RM-4.9 Type E refuses to invent an envelope
  [PASS] BP-5.1 Type 2-A curb profile closes — h = 6 in
  [PASS] BP-5.1 catalog resolves every sheet label
  [PASS] RM-4.2 32 in PCB cataloged
  [PASS] RM-4.1 50->32 transition is 6 ft


## RM-4.6 — Concrete Barrier End Sections & RM-4.4 — Single Slope Barrier Transitions

RM-4.6 steps a Type B / B1 / D barrier down to the 32 in vertical-faced
end a Bridge Terminal Assembly (MGS-3.x) or impact attenuator bolts to —
`layout_barrier_end_section` returns the lofting stations. RM-4.4 turned
out to be **plan-width** transitions (40 ft tapers wrapping sign-support
foundations and pier columns), not profile-to-profile lofts —
`layout_barrier_transition` returns width-vs-station.

In [21]:
from civilpy.structural.odot.roadway_barrier import (
    BARRIER_END_SECTIONS, layout_barrier_end_section, layout_barrier_transition)

for des in ("Type B", "Type B1", "Type D"):
    lay = layout_barrier_end_section(des)
    xs = [x for x, _ in lay.stations]
    h0 = max(z for _, z in lay.stations[0][1])
    hN = max(z for _, z in lay.stations[-1][1])
    print(f"  {des:8s}: {xs[-1]:5.1f} ft, {h0:g} in -> {hN:g} in tall, {len(lay.stations)} stations")

b = layout_barrier_end_section("Type B")
check("RM-4.6 Type B runs 30 ft to a 32 in end",
      b.stations[-1][0] == 30.0 and max(z for _, z in b.stations[-1][1]) == 32.0)
b1 = layout_barrier_end_section("Type B1")
check("RM-4.6 Type B1 tapers 57 -> 42 in over the 16 ft body",
      max(z for _, z in b1.stations[1][1]) == 42.0)

t = layout_barrier_transition("Type B", "sign support", obstruction_width_in=48.0)
check("RM-4.4 sign-support transition: 40 + 10 + 40 ft, 12 -> 48 in wide",
      t.total_length_ft == 90.0 and [w for _, w in t.stations] == [12.0, 48.0, 48.0, 12.0])
p = layout_barrier_transition("Type B1", "pier", obstruction_length_ft=14.0)
check("RM-4.4 pier protection carries 5 ft each side of the columns",
      p.total_length_ft == 40 + 5 + 14 + 5 + 40)
report["barrier_end_sections"] = len(BARRIER_END_SECTIONS)
report["barrier_transitions"] = 2


  Type B  :  30.0 ft, 42 in -> 32 in tall, 5 stations
  Type B1 :  30.0 ft, 57 in -> 32 in tall, 5 stations
  Type D  :  14.0 ft, 42 in -> 32 in tall, 4 stations
  [PASS] RM-4.6 Type B runs 30 ft to a 32 in end
  [PASS] RM-4.6 Type B1 tapers 57 -> 42 in over the 16 ft body
  [PASS] RM-4.4 sign-support transition: 40 + 10 + 40 ft, 12 -> 48 in wide
  [PASS] RM-4.4 pier protection carries 5 ft each side of the columns


## RM-5.2 — Bikeway Railing

A treated-wood post-and-rail fence along bike paths (6x6 posts, 2x8 top
rail, two 2x12 face rails, 42 in overall) — **not** a crashworthy
barrier, and a wood system rather than the BR-2-15 steel-tube pairing
the wave planning guessed.

In [22]:
from civilpy.structural.odot.bikeway_railing import (
    BikewayRailingInput, layout_bikeway_railing, RAILING_HEIGHT_IN)

lay = layout_bikeway_railing(BikewayRailingInput(100.0))
print(f"  100 ft run + flares: {lay.total_length_ft:g} ft total, "
      f"{len(lay.post_stations_ft)} posts, {len(lay.midspan_stations_ft)} mid-span stiffeners, "
      f"{lay.n_rail_pieces} rail pieces")
check("RM-5.2 flared ends add 20 ft each", lay.total_length_ft == 140.0)
gaps = [b - a for a, b in zip(lay.post_stations_ft, lay.post_stations_ft[1:])]
check("RM-5.2 posts never exceed 10 ft centers", max(gaps) <= 10.0 + 1e-9)
check("RM-5.2 rail top at 42 in", RAILING_HEIGHT_IN == 42.0)
low = layout_bikeway_railing(BikewayRailingInput(60.0, flared_ends=False, low_shoulder=True))
check("RM-5.2 low-shoulder posts embed 5 ft", low.embedment_in == 60.0)
report["bikeway_railing"] = len(lay.post_stations_ft)


  100 ft run + flares: 140 ft total, 15 posts, 14 mid-span stiffeners, 21 rail pieces
  [PASS] RM-5.2 flared ends add 20 ft each
  [PASS] RM-5.2 posts never exceed 10 ft centers
  [PASS] RM-5.2 rail top at 42 in
  [PASS] RM-5.2 low-shoulder posts embed 5 ft


## RM-4.7 — Thrie-Beam Transition for Portable Concrete Barrier

A nested 6'-3" 12-gauge thrie-beam bridging the ≤ 1 ft gap between two
32 in PCB shape families, one connection pair per sheet. The J-J Hook
NJ shape has **no** approved pair — the lookup must refuse it.

In [23]:
from civilpy.structural.odot.roadway_portable_barrier import (
    THRIE_BEAM_PCB_TRANSITIONS, thrie_beam_pcb_transition, thrie_beam_transition_notes)

for (a, b), t in THRIE_BEAM_PCB_TRANSITIONS.items():
    print(f"  sheet {t.sheet}: {a}  <->  {b}")
t1 = thrie_beam_pcb_transition('Generic 32" F-shape PCB', 'Generic 32" New Jersey shape PCB')
check("RM-4.7 lookup is order-free", t1.sheet == 1)
try:
    thrie_beam_pcb_transition('J-J Hook 32" New Jersey shape PCB', 'Generic 32" F-shape PCB')
    check("RM-4.7 refuses the J-J Hook NJ shape", False)
except ValueError:
    check("RM-4.7 refuses the J-J Hook NJ shape", True)
notes = " ".join(thrie_beam_transition_notes())
check("RM-4.7 deployment limits carried", "once per mile" in notes and "100 ft" in notes)
report["pcb_thrie_transitions"] = len(THRIE_BEAM_PCB_TRANSITIONS)


  sheet 1: Generic 32" New Jersey shape PCB  <->  Generic 32" F-shape PCB
  sheet 2: Generic 32" New Jersey shape PCB  <->  J-J Hook 32" F-shape PCB
  sheet 3: Generic 32" F-shape PCB  <->  J-J Hook 32" F-shape PCB
  [PASS] RM-4.7 lookup is order-free
  [PASS] RM-4.7 refuses the J-J Hook NJ shape
  [PASS] RM-4.7 deployment limits carried


## VPF-1-24 — Vandal Protection Fence

Chain-link fence on structures; posts at the section's tabulated max
spacing with top/bottom rail lines.

In [24]:
from civilpy.structural.odot.vandal_fence import (
    POST_SECTIONS, FenceRunInput, layout_fence_run)

fence = layout_fence_run(FenceRunInput(120.0))
print(f"  {fence.inputs.post_name}: {len(fence.post_stations_ft)} posts over 120 ft, "
      f"height {fence.section.height_ft:g} ft")
spacings = [b - a for a, b in zip(fence.post_stations_ft, fence.post_stations_ft[1:])]
check("VPF-1-24 spacing never exceeds the tabulated max",
      max(spacings) <= fence.section.max_spacing_ft + 1e-9)
check("VPF-1-24 rails span the full run", fence.top_rail[1][0] == 120.0)
report["vandal_fence"] = len(POST_SECTIONS)


  PS-2/BP-1: 13 posts over 120 ft, height 6 ft
  [PASS] VPF-1-24 spacing never exceeds the tabulated max
  [PASS] VPF-1-24 rails span the full run


## MGS — Guardrail Runs & Bridge Terminal Assemblies (MGS-2.1, MGS-3.1/3.2/3.3)

The standard-run layout plus the three bridge terminal assemblies as
post-by-post layouts — the hardware that connects a guardrail run to
the Wave-3 bridge railings and the RM-4.6 barrier end sections.

In [25]:
from civilpy.structural.odot.guardrail import (
    BRIDGE_TERMINALS, MGS_DRAWINGS, layout_bridge_terminal, layout_mgs_run,
    terminals_for_railing)

run = layout_mgs_run(150.0)
print(f"  standard run: {len(run.post_stations_ft)} posts @ 6'-3\", "
      f"{run.n_panels} x {run.panel_length_ft:g} ft panels, rail at {run.rail_height_in:g} in")
check("MGS-2.1 run posts at 6.25 ft", run.post_stations_ft[1] == 6.25)

for des in ("Type 1", "Type 2", "Type TST-2"):
    lay = layout_bridge_terminal(des)
    t = lay.terminal
    print(f"  {t.scd} {des:10s}: {t.n_posts:2d} posts over {lay.length_in / 12.0:5.2f} ft "
          f"from the {t.origin}")
t1 = layout_bridge_terminal("Type 1")
check("MGS-3.1 Type 1: 13 posts, long posts 1-6",
      len(t1.posts) == 13 and all(l == 78.0 for n, _, l, _ in t1.posts if n <= 6))
check("MGS-3.3 TST-2 mates with TST-2-21",
      any(d.scd == "MGS-3.3" for d in terminals_for_railing("TST-2-21")))
check("MGS registry notes cover the rated-3 sheets",
      all(MGS_DRAWINGS[s].notes for s in ("MGS-2.3", "MGS-4.1", "MGS-4.2", "MGS-6.1")))
report["mgs_terminals"] = len(BRIDGE_TERMINALS)


  standard run: 25 posts @ 6'-3", 6 x 25 ft panels, rail at 31 in
  [PASS] MGS-2.1 run posts at 6.25 ft
  MGS-3.1 Type 1    : 13 posts over 25.33 ft from the parapet / barrier end
  MGS-3.2 Type 2    :  1 posts over  3.12 ft from the trailing end of parapet / barrier
  MGS-3.3 Type TST-2: 10 posts over 21.88 ft from the MGS end (last standard-run post)
  [PASS] MGS-3.1 Type 1: 13 posts, long posts 1-6
  [PASS] MGS-3.3 TST-2 mates with TST-2-21
  [PASS] MGS registry notes cover the rated-3 sheets


## PSID-1-13 — Prestressed I-Beams: catalog, strand designer, MIDAS spoke

All 13 sections (AASHTO 2/3/4, three Modified Type 4s with their true
36/36/48 in top flanges, and the 7 WF sections), the vector-extracted
permissible strand grids, and the pipeline that **designs** the strand
pattern — there is no PSIDD companion sheet to verify against, so the
line checks are the design source.

In [26]:
from civilpy.structural.odot.ps_i_beam import (
    PS_I_BEAM_SECTIONS, ps_i_beam_profile, ps_i_beam_section, strand_grid)
from civilpy.structural.ps_i_beam_pipeline import (
    ps_i_beam_line_checks, structural_model_from_ps_i)

check("PSID-1-13 catalogs 13 sections", len(PS_I_BEAM_SECTIONS) == 13)
for name, s in PS_I_BEAM_SECTIONS.items():
    n = sum(len(ys) for _, ys in s.strand_rows)
    assert n == s.max_bottom_flange_strands, name
check("every strand grid reconciles with the sheet's permissible count", True)
check("Modified Type 4 (72in) carries the 48 in top flange",
      ps_i_beam_section("Modified AASHTO Type 4 (72in)").top_flange_width_in == 48.0)
prof = ps_i_beam_profile("WF48-49")
check("WF48-49 outline spans 49 in x 48 in",
      approx(max(y for y, _ in prof) - min(y for y, _ in prof), 49.0)
      and approx(max(z for _, z in prof), 48.0))

psi = ps_i_beam_line_checks("WF48-49", 95.0, 5, spacing_ft=9.0,
                            barrier_klf=0.9, fci_ksi=5.0, fc_ksi=7.0)
print(psi.summary())
check("WF48-49 @ 95 ft designs a passing line", psi.all_ok)
check("long spans engage end debonding", psi.design.n_debonded > 0)

m = structural_model_from_ps_i("WF48-49", 95.0, 5, spacing_ft=9.0, barrier_klf=0.9)
girders = [e for e in m.elements.values() if e.role == "girder"]
check("MIDAS spoke breaks each line at the 3 quarter-point diaphragms",
      len(girders) == 5 * 4)
report["ps_i_beam"] = len(PS_I_BEAM_SECTIONS)


  [PASS] PSID-1-13 catalogs 13 sections
  [PASS] every strand grid reconciles with the sheet's permissible count
  [PASS] Modified Type 4 (72in) carries the 48 in top flange
  [PASS] WF48-49 outline spans 49 in x 48 in


WF48-49 @ 95 ft, S = 9 ft: 36 x 0.6 in strands, e = 20.61 in (straight, 8 debonded each end)
  DF moment 0.750 / shear 0.884 (type k, 4.6.2.2.2b/3a)
  losses: ES 17.9 + LT 25.9 ksi -> f_pe = 158.6 ksi
  PASS  transfer compression: D/C = 0.86 (5.9.2.3.1a)
  PASS  transfer tension: D/C = 0.92 (5.9.2.3.1b)
  PASS  service compression: D/C = 0.52 (5.9.2.3.2a)
  PASS  service III tension: D/C = 0.99 (5.9.2.3.2b)
  PASS  Strength I flexure: D/C = 0.86 (5.6.3.2.2)
  release camber: +2.38 in (elastic estimate)
  [PASS] WF48-49 @ 95 ft designs a passing line
  [PASS] long spans engage end debonding
  [PASS] MIDAS spoke breaks each line at the 3 quarter-point diaphragms


## BrIM emit — steel girder bridge (deck, parapets, rebar, studs, bearings)

The full Phase 1-3 superstructure emit: the crowned closed deck solid,
BDM 309-4 overhangs and haunches, the SBR-1-20 parapet with its bar
cage, shear studs, and the bearing stacks — everything
`draw_bim_emit.py` renders in Rhino, verified here as the neutral
`EmitObject` record plus the pay-item rollup.

In [27]:
import collections
from civilpy.structural.bridge_layout import BridgeInput
from civilpy.structural.rhino_bim import girder_bridge_emit, pay_item_quantities, emit_to_json

inp = BridgeInput(spans_ft=(80.0, 80.0), girder_count=4, girder_spacing_ft=9.0,
                  girder_label="W36X150", overhang_ft=2.5, railing="SBR-1-20")
emit = girder_bridge_emit(inp)
by_type = collections.Counter(o.tags.get("bim.type") for o in emit.objects)
for t, n in sorted(by_type.items(), key=lambda kv: str(kv[0])):
    print(f"  {str(t):12s}: {n}")
for t in ("bridge", "deck", "girder", "haunch", "parapet", "rebar",
          "shear_stud", "bearing", "load_plate"):
    check(f"steel emit carries {t} objects", by_type.get(t, 0) >= 1)
check("one crowned deck solid", by_type["deck"] == 1)
check("4 girder prisms", by_type["girder"] == 4)

q = pay_item_quantities(emit)
for code_, d in sorted(q.items()):
    print(f"  {code_}: {d['qty']:10.1f} {d['unit']:3s} {d['desc'][:52]}")
check("steel + studs + concrete + rebar items roll up",
      {"513E10220", "513E20000", "511E12100"} <= set(q))
check("emit serializes for the Rhino driver", len(emit_to_json(emit)) > 1000)
report["steel_girder_emit"] = len(emit.objects)


  None        : 16
  bearing     : 12
  bridge      : 1
  deck        : 1
  girder      : 4
  haunch      : 4
  load_plate  : 12
  parapet     : 2
  rebar       : 2073
  shear_stud  : 960
  [PASS] steel emit carries bridge objects
  [PASS] steel emit carries deck objects
  [PASS] steel emit carries girder objects
  [PASS] steel emit carries haunch objects
  [PASS] steel emit carries parapet objects
  [PASS] steel emit carries rebar objects
  [PASS] steel emit carries shear_stud objects
  [PASS] steel emit carries bearing objects
  [PASS] steel emit carries load_plate objects
  [PASS] one crowned deck solid
  [PASS] 4 girder prisms
  509E00200:    41627.1 lb  Epoxy coated reinforcing steel [CONFIRM]
  509E00300:        0.0 lb  GFRP deformed bars [CONFIRM]
  511E12100:      142.2 cy  Class QC2 concrete, superstructure (deck) [CONFIRM]
  512E10000:       48.4 cy  Concrete, parapet/railing [CONFIRM]
  513E10220:    98250.9 lb  Structural steel members, Level 1
  513E20000:      960.0 ea  S

## BrIM emit — adjacent box beams (PSBD/PSBDD standard designs)

The Phase-5 slice: L1 line checks re-derive the PSBDD-1-25 tabulated
design's governing checks, then the emit draws the hollow members,
strand rows, tie rods, diaphragms, pads, and composite topping.

In [28]:
from civilpy.structural.box_beam_pipeline import box_beam_line_checks
from civilpy.structural.rhino_box_bim import BoxBridgeInput, box_beam_bridge_emit

bb = box_beam_line_checks("CB27-48", 60.0, 9, barrier_klf=0.9)
print(bb.summary())
check("CB27-48 @ 60 ft standard design verifies", bb.all_ok)

bemit = box_beam_bridge_emit(BoxBridgeInput(box="CB27-48", span_ft=60.0, n_beams=9))
bq = pay_item_quantities(bemit)
check("box members count 9 each into the 515 item", bq["515E10000"]["qty"] == 9)
check("composite topping measures into 511", "511E12100" in bq)
report["box_beam_emit"] = len(bemit.objects)


CB27-48 @ 60 ft (composite), 24 strands, e = 10.83 in:
  DF moment 0.295 / shear 0.458 (adjacent box, 4.6.2.2.2b/3c)
  losses: ES 12.2 + LT 24.8 ksi -> f_pe = 165.5 ksi
  PASS  transfer compression: D/C = 0.92 (5.9.2.3.1a)
  PASS  transfer tension: D/C = 0.93 (5.9.2.3.1b)
  PASS  service compression: D/C = 0.34 (5.9.2.3.2a)
  PASS  service III tension: D/C = 0.00 (5.9.2.3.2b)
  PASS  Strength I flexure: D/C = 0.64 (5.6.3.2.2)
  camber: 1.125 in release, 1.875 in erection (tabulated)
  [PASS] CB27-48 @ 60 ft standard design verifies
  [PASS] box members count 9 each into the 515 item
  [PASS] composite topping measures into 511


## BrIM emit — prestressed I-beams (the executed design drives the drawing)

The Phase-6 slice: the same `PSIBeamLineChecks` result from the designer
section above feeds the emit, so the drawn strand rows *are* the
designed pattern (with the end-debond counts riding in the tags).

In [29]:
from civilpy.structural.rhino_ps_i_bim import PSIBridgeInput, ps_i_bridge_emit

pemit = ps_i_bridge_emit(
    PSIBridgeInput("WF48-49", 95.0, 5, spacing_ft=9.0, overhang_ft=2.5,
                   fci_ksi=5.0, fc_ksi=7.0, barrier_klf=0.9),
    checks=psi)
pq = pay_item_quantities(pemit)
for code_, d in sorted(pq.items()):
    print(f"  {code_}: {d['qty']:8.1f} {d['unit']:3s} {d['desc'][:52]}")
check("5 I-beam members into 515E20000", pq["515E20000"]["qty"] == 5)
rows = [o for o in pemit.of_type("tendon") if o.tags["bim.id"].startswith("PSI1-")]
check("drawn strand rows sum to the designed count",
      sum(int(o.tags["tendon.strands"]) for o in rows) == psi.design.n_strands)
check("debonded strands ride in the tendon tags",
      sum(int(o.tags.get("tendon.debonded", 0)) for o in rows) == psi.design.n_debonded)
report["ps_i_emit"] = len(pemit.objects)


  511E12100:    114.2 cy  Class QC2 concrete, superstructure (deck) [CONFIRM]
  515E20000:      5.0 ea  Prestressed concrete I-beam member [CONFIRM]
  515E30000:     12.0 ea  Intermediate diaphragms [CONFIRM]
  516E10000:     10.0 ea  Elastomeric bearing [CONFIRM]
  [PASS] 5 I-beam members into 515E20000
  [PASS] drawn strand rows sum to the designed count
  [PASS] debonded strands ride in the tendon tags


## Substructure type gallery — Phase 4 / 4v

Every substructure type placed under the same two-span steel layout and
emitted: the Phase-4 multi-column bent + seat abutments, then a mixed
assembly (integral abutment / hammerhead pier / semi-integral abutment)
and a capped-pile (pile bent) variant. The cap designs here are compact
stand-ins shaped like `optimize_pier_cap` results — the real design
loop is the Substructure notebook's job.

In [30]:
from types import SimpleNamespace
from civilpy.structural.abutment import RetainingWall
from civilpy.structural.aashto.lrfd.columns import RebarLayer
from civilpy.structural.bridge_layout import layout_bridge
from civilpy.structural.pier import MultiColumnBent, PierCap, PierColumn
from civilpy.structural.stm_topology.design import DepthCandidate, PierCapDesign
from civilpy.structural.substructure_layout import (
    AbutmentSpec, BentPierSpec, FootingSpec, HammerheadSpec,
    IntegralAbutmentSpec, PileBentSpec, SeatAbutmentSpec,
    SemiIntegralAbutmentSpec, assemble_substructure, substructure_from_layout)
from civilpy.structural.rhino_bim import substructure_emit


def cap_design(span, depth, thickness, *, tie_force=700.0, bar_size=10,
               bar_count=12, tie_at_top=False):
    tie_y = depth - 0.2 if tie_at_top else 0.2
    tie = SimpleNamespace(force=tie_force, bar_size=bar_size,
                          bar_count=bar_count, member=("A", "B"))
    model = SimpleNamespace(nodes={"A": (0.0, tie_y), "B": (span, tie_y)})
    result = SimpleNamespace(report=SimpleNamespace(ties=[tie]), model=model)
    cand = DepthCandidate(depth=depth, cost=1.0, concrete_cost=1.0,
                          steel_lb=100.0, strut_angle=40.0, node_ratio=2.0,
                          max_tie=tie_force, complete=True, feasible=True,
                          result=result)
    return PierCapDesign(optimal=cand, candidates=[cand], span=span,
                         thickness=thickness)


layout = layout_bridge(inp)          # the steel-girder layout from above
pier_cap = cap_design(32.0, 5.0, 4.0)
abut_cap = cap_design(32.0, 3.5, 3.0, tie_force=250.0, bar_size=8, bar_count=6)
column = PierColumn(height=240.0, diameter=42.0,
                    layers=[RebarLayer(area=6.0, depth=6.0),
                            RebarLayer(area=6.0, depth=36.0)])
bent = MultiColumnBent(
    PierCap(length=32.0 * 12.0, width=48.0, depth=60.0,
            column_positions=[138.0, 246.0]),
    [column, column])
wall = RetainingWall(stem_height=14.0, stem_thickness=1.5, toe_length=4.0,
                     heel_length=8.0, footing_thickness=3.0,
                     backfill_gamma=120.0, backfill_phi=32.0)
abut_spec = AbutmentSpec(pile_xs_ft=(1.5, 10.0, 18.5, 27.0),
                         pile_shape="HP10X42", pile_length_ft=40.0,
                         wingwall=wall, wingwall_length_ft=12.0)

# Phase 4: bent pier + seat abutments
sub = substructure_from_layout(
    layout, pier_cap=pier_cap, pier_bent=bent, abutment_cap=abut_cap,
    abutment=abut_spec,
    footing=FootingSpec(length_ft=10.0, width_ft=10.0, thickness_ft=3.0))
objs = substructure_emit(sub)
check("Phase 4 bent-pier substructure emits",
      len(sub.piers) == 1 and len(sub.abutments) == 2 and len(objs) > 100,
      f"{len(objs)} tagged objects")

# Phase 4v: pile bent piers under the same seat abutments
sub_pb = assemble_substructure(layout, {
    "pier": PileBentSpec(cap_design=pier_cap,
                         pile_xs_ft=(2.0, 9.0, 16.0, 23.0, 30.0)),
    "abutment": SeatAbutmentSpec(cap_design=abut_cap, spec=abut_spec)})
check("pile bent (capped-pile pier) places and emits",
      len(sub_pb.piers[0].piles) > 0 and not sub_pb.piers[0].columns
      and len(substructure_emit(sub_pb)) > 50)

# Phase 4v: integral / hammerhead / semi-integral mix
sub_mix = assemble_substructure(layout, {
    0: IntegralAbutmentSpec(pile_xs_ft=(2.0, 10.0, 17.0, 25.0)),
    1: HammerheadSpec(cap_design=cap_design(32.0, 6.0, 5.0, tie_at_top=True),
                      column=column, tip_depth_ft=3.0,
                      footing=FootingSpec(14.0, 14.0, 3.5)),
    2: SemiIntegralAbutmentSpec(cap_design=abut_cap, spec=abut_spec)})
kinds = [a.kind for a in sub_mix.abutments]
print(f"  mixed assembly: abutments {kinds}, "
      f"pier columns {[len(p.columns) for p in sub_mix.piers]}")
check("mixed types place on one bridge",
      set(kinds) == {"integral", "semi-integral"}
      and len(sub_mix.piers[0].columns) == 1)   # hammerhead stem
mix_objs = substructure_emit(sub_mix)
by_type_sub = collections.Counter(o.tags.get("bim.type") for o in mix_objs)
print("  mixed emit:", dict(sorted((str(k), v) for k, v in by_type_sub.items())))
check("integral abutment emits piles but no beam seats",
      len(sub_mix.abutments[0].piles) > 0 and len(sub_mix.abutments[0].seats) == 0)
report["substructure_types"] = 5


  [PASS] Phase 4 bent-pier substructure emits — 677 tagged objects
  [PASS] pile bent (capped-pile pier) places and emits
  mixed assembly: abutments ['integral', 'semi-integral'], pier columns [1]
  [PASS] mixed types place on one bridge
  mixed emit: {'abutment_cap': 1, 'beam_seat': 8, 'column': 1, 'diaphragm': 2, 'footing': 1, 'pier_cap': 1, 'pile': 8, 'rebar': 412, 'wingwall': 4}
  [PASS] integral abutment emits piles but no beam seats


## Barrier generation — every shape family to a tagged `.3dm`

`rhino_barrier.build_barriers` renders the actual ODOT standard section
for any catalog entry. One representative of **each shape family** (New
Jersey, single slope, combination, steel post-and-beam, freestanding
portable/median) is written to a real `.3dm` and read back through the
same tag round-trip the `DeckBarrier` importer uses — plus the lane
markings. This is the front half of the Grasshopper integration work:
the same files these calls produce are what the GH components will
consume.

In [31]:
import os, tempfile, warnings
import rhino3dm
from civilpy.structural.rhino_gdr import GTAG
from civilpy.structural.rhino_barrier import (
    build_barriers, build_lane_lines, read_barrier_model)


def author_girders(path, ys=(0.0, 7.0, 14.0, 21.0, 28.0)):
    f = rhino3dm.File3dm()
    f.Settings.ModelUnitSystem = rhino3dm.UnitSystem.Feet
    for i, y in enumerate(ys, start=1):
        pl = rhino3dm.Polyline()
        for x in (0.0, 60.0, 120.0):
            pl.Add(x, y, 0.0)
        ga = rhino3dm.ObjectAttributes()
        for k, v in (("kind", "girder"), ("shape", "W24X104"),
                     ("grade", "Grade 50"), ("line", i)):
            ga.SetUserString(GTAG + k, str(v))
        f.Objects.AddCurve(pl.ToPolylineCurve(), ga)
    assert f.Write(str(path), 7)


tmp = tempfile.mkdtemp()
gpath = os.path.join(tmp, "girders.3dm")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    author_girders(gpath)

FAMILIES = [
    ("BR-1 (36 in)", None),                              # new jersey, edges
    ("SBR-1 (42 in)", None),                             # single slope, edges
    ("BR-2 (sidewalk barrier + twin tube)", None),       # combination
    ("TST-2 (three steel tube)", None),                  # steel post-and-beam
    ("PCB (portable, unanchored)", "median"),            # freestanding portable
    ("SBR-2 (57 in median)", "median"),                  # median single slope
]
for des, place in FAMILIES:
    out = os.path.join(tmp, des.replace(" ", "_").replace("/", "-") + ".3dm")
    model = build_barriers(gpath, out_path=out, designation=des,
                           placements=place)
    back = read_barrier_model(out)
    kinds = collections.Counter(o["kind"] for o in back)
    print(f"  {des:40s} {model.shape_family:14s} dc2 = {model.total_dc2_klf:6.3f} klf "
          f"-> {kinds['barrier']} barrier / {kinds['rebar']} rebar objects")
    check(f"{des}: builds + round-trips", kinds["barrier"] >= 1)

lanes = build_lane_lines(gpath, out_path=os.path.join(tmp, "lanes.3dm"))
check("lane markings paint edge + divider lines",
      lanes.n_edge_lines == 2 and lanes.n_divider_lines >= 1,
      f"{lanes.n_lanes} lanes over {lanes.usable_width_ft:.1f} ft")
report["barrier_families"] = len(FAMILIES)


  BR-1 (36 in)                             new jersey     dc2 =  0.882 klf -> 2 barrier / 246 rebar objects
  [PASS] BR-1 (36 in): builds + round-trips
  SBR-1 (42 in)                            single slope   dc2 =  1.225 klf -> 2 barrier / 248 rebar objects
  [PASS] SBR-1 (42 in): builds + round-trips


C:\Users\dane\PycharmProjects\civilpy\src\civilpy\structural\rhino_gdr.py:670: UserWarning: girder model is missing gdr.deck_t and/or gdr.deck_weff -- the composite deck section cannot be built without them; supply the structural slab thickness (in) and effective width (in), or fall back to ODOT BDM defaults downstream (G5).
  warnings.warn(


  BR-2 (sidewalk barrier + twin tube)      combination    dc2 =  0.000 klf -> 42 barrier / 248 rebar objects
  [PASS] BR-2 (sidewalk barrier + twin tube): builds + round-trips
  TST-2 (three steel tube)                 steel tube     dc2 =  0.160 klf -> 38 barrier / 0 rebar objects
  [PASS] TST-2 (three steel tube): builds + round-trips
  PCB (portable, unanchored)               portable       dc2 =  0.000 klf -> 1 barrier / 123 rebar objects
  [PASS] PCB (portable, unanchored): builds + round-trips
  SBR-2 (57 in median)                     single slope   dc2 =  1.357 klf -> 1 barrier / 65 rebar objects
  [PASS] SBR-2 (57 in median): builds + round-trips
  [PASS] lane markings paint edge + divider lines — 3 lanes over 32.0 ft


## Summary

In [32]:
print("All SCD components exercised successfully:")
for k, v in report.items():
    print(f"  {k:24s}: {v}")
print(f"\n{len(report)} components checked, 0 failures (any assertion failure would have stopped this notebook above).")


All SCD components exercised successfully:
  A-1-20                  : 1
  AS-1-15                 : 266
  DS-1-92                 : 7
  PCB-91                  : 4
  AS-2-15                 : 33
  HW-2.1/HW-2.2           : 2
  HW-1.1                  : 2
  BCHW                    : 3
  bridge_railing          : 14
  SB-1-24                 : 159
  CS-1-24                 : 169
  CPA-1-08                : 6
  CPP-1-08                : 6
  RB-1-55                 : 1
  FB-1-82                 : 1
  BD-1-11                 : 1
  EXJ-4-87/EXJ-5-93       : 9
  rhino_layers            : 3
  roadway_barrier         : 7
  concrete_curb           : 19
  roadway_pcb             : 2
  barrier_end_sections    : 3
  barrier_transitions     : 2
  bikeway_railing         : 15
  pcb_thrie_transitions   : 3
  vandal_fence            : 3
  mgs_terminals           : 3
  ps_i_beam               : 13
  steel_girder_emit       : 3085
  box_beam_emit           : 87
  ps_i_emit               : 54
  substruct